In [2]:
import pandas as pd
import numpy as np
import pickle
import os

# Load raw data
ratings = pd.read_csv('../data/ml-25m/ratings.csv')
movies = pd.read_csv('../data/ml-25m/movies.csv')

# Temporal split — same as training
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
train = ratings[ratings['datetime'] < '2015-01-01'].copy()

# Normalize ratings
min_rating = 0.5
max_rating = 5.0
train['rating'] = (train['rating'] - min_rating) / (max_rating - min_rating)

# Compute movie stats from training data only
movie_stats = train.groupby('movieId')['rating'].agg(['mean', 'count'])
movie_stats.columns = ['avg_rating', 'num_ratings']
movie_stats['avg_rating'] = (movie_stats['avg_rating'] * 
                              (max_rating - min_rating)) + min_rating

max_num_ratings = int(movie_stats['num_ratings'].max())

# Clean movies dataframe
movies_clean = movies[['movieId', 'title', 'genres']].copy()
movies_clean = movies_clean[
    movies_clean['genres'] != '(no genres listed)'].reset_index(drop=True)

# Save lightweight data file
data_bundle = {
    'movies_df': movies_clean,
    'movie_stats': movie_stats,
    'max_num_ratings': max_num_ratings
}

with open('../models/movie_data.pkl', 'wb') as f:
    pickle.dump(data_bundle, f)

print(f"Movies: {len(movies_clean):,}")
print(f"Movie stats: {len(movie_stats):,}")
print(f"Max ratings: {max_num_ratings:,}")
print(f"Saved to ../models/movie_data.pkl")

size = os.path.getsize('../models/movie_data.pkl')
print(f"File size: {size/1024/1024:.2f} MB")

Movies: 57,361
Movie stats: 22,316
Max ratings: 59,021
Saved to ../models/movie_data.pkl
File size: 2.83 MB


In [ ]:

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pickle
import re
import numpy as np
import torch
import torch.nn as nn
import gradio as gr

# ── Paths ─────────────────────────────────────────────────────
MODEL_DIR = '../models'

# ── Load lightweight movie data (no raw ratings needed) ───────
with open(f'{MODEL_DIR}/movie_data.pkl', 'rb') as f:
    data = pickle.load(f)

movies_df       = data['movies_df']        # movieId, title, genres
movie_stats     = data['movie_stats']      # avg_rating, num_ratings
max_num_ratings = data['max_num_ratings']

# ── Load GMF model info (contains user/movie index mappings) ──
with open(f'{MODEL_DIR}/gmf_regularized_info.pkl', 'rb') as f:
    model_info = pickle.load(f)

user_id_to_index  = model_info['user_id_to_index']
movie_id_to_index = model_info['movie_id_to_index']
N_USERS    = model_info['n_users']
N_MOVIES   = model_info['n_movies']
EMB_DIM    = model_info['embedding_dim']
DROPOUT    = model_info['dropout']
MIN_RATING = model_info['min_rating']
MAX_RATING = model_info['max_rating']

# ── GMF model ─────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class GMF(nn.Module):
    def __init__(self, n_users, n_movies, embedding_dim=32, dropout=0.3):
        super().__init__()
        self.user_embedding  = nn.Embedding(n_users, embedding_dim)
        self.movie_embedding = nn.Embedding(n_movies, embedding_dim)
        self.dropout         = nn.Dropout(p=dropout)
        self.bn              = nn.BatchNorm1d(embedding_dim)
        self.output_layer    = nn.Linear(embedding_dim, 1)

    def forward(self, user_indices, movie_indices):
        u = self.user_embedding(user_indices)
        m = self.movie_embedding(movie_indices)
        return self.output_layer(self.dropout(self.bn(u * m))).squeeze()

gmf_model = GMF(N_USERS, N_MOVIES, EMB_DIM, DROPOUT).to(device)
gmf_model.load_state_dict(
    torch.load(f'{MODEL_DIR}/gmf_regularized_best.pth', map_location=device))
gmf_model.eval()

# ── Genre setup ───────────────────────────────────────────────
ALL_GENRES = [
    'Action','Adventure','Animation','Children','Comedy',
    'Crime','Documentary','Drama','Fantasy','Film-Noir',
    'Horror','IMAX','Musical','Mystery','Romance',
    'Sci-Fi','Thriller','War','Western',
]

GENRE_COLORS = {
    'Action':'#ef4444','Adventure':'#f97316','Animation':'#ec4899',
    'Children':'#06b6d4','Comedy':'#eab308','Crime':'#991b1b',
    'Documentary':'#64748b','Drama':'#8b5cf6','Fantasy':'#14b8a6',
    'Film-Noir':'#475569','Horror':'#7c3aed','IMAX':'#3b82f6',
    'Musical':'#d946ef','Mystery':'#92400e','Romance':'#f43f5e',
    'Sci-Fi':'#0891b2','Thriller':'#ea580c','War':'#65a30d',
    'Western':'#d97706',
}

def _make_genre_vec(genres_str):
    v = np.zeros(len(ALL_GENRES), dtype=np.float32)
    for g in genres_str.split('|'):
        if g in ALL_GENRES:
            v[ALL_GENRES.index(g)] = 1.0
    return v

movie_id_to_genre = {row.movieId: _make_genre_vec(row.genres)
                     for row in movies_df.itertuples()}
movie_title_map   = dict(zip(movies_df['movieId'], movies_df['title']))
movie_genre_map   = dict(zip(movies_df['movieId'], movies_df['genres']))

print(f"Loaded  {len(movies_df):,} movies  |  "
      f"{N_USERS:,} users  |  device={device}")

# ── HTML helpers ──────────────────────────────────────────────
def _stars(rating):
    n = max(0, min(5, round(rating)))
    return '★' * n + '☆' * (5 - n)

def _year(title):
    m = re.search(r'\((\d{4})\)\s*$', title)
    return m.group(1) if m else ''

def _clean(title):
    return re.sub(r'\s*\(\d{4}\)\s*$', '', title).strip()

def _badges(genres_str):
    parts = []
    for g in genres_str.split('|'):
        c = GENRE_COLORS.get(g, '#6b7280')
        parts.append(
            f'<span style="background:{c};color:#fff;padding:2px 9px;'
            f'border-radius:20px;font-size:11px;font-weight:600;'
            f'margin:2px 2px 2px 0;display:inline-block;">{g}</span>'
        )
    return ''.join(parts)

def _rank_color(rank):
    return {1:'#f59e0b', 2:'#94a3b8', 3:'#b45309'}.get(rank, '#4b5563')

def build_cards(recs, score_key='combined_score'):
    if not recs:
        return ('<div style="text-align:center;color:#94a3b8;padding:48px;">'
                'No results found. Try adjusting your filters.</div>')

    cards = ['<div style="display:flex;flex-direction:column;gap:10px;padding:4px 0;">']

    for rec in recs:
        rank       = rec['rank']
        title_raw  = rec['title']
        genres_str = rec.get('genres', '')
        avg_r      = rec['avg_rating']
        num_r      = rec['num_ratings']
        score      = rec.get(score_key, 0.0)

        title  = _clean(title_raw)
        year   = _year(title_raw)
        stars  = _stars(avg_r)
        badges = _badges(genres_str)
        num_fmt = f'{int(num_r):,}'

        cards.append(f"""
<div style="background:#1a1f35;border:1px solid #2d3448;border-radius:14px;
            padding:16px 18px;display:flex;align-items:flex-start;gap:14px;">

  <!-- Rank badge -->
  <div style="min-width:42px;height:42px;border-radius:50%;
              background:{_rank_color(rank)};display:flex;
              align-items:center;justify-content:center;
              font-size:17px;font-weight:800;color:#fff;flex-shrink:0;">{rank}</div>

  <!-- Main content -->
  <div style="flex:1;min-width:0;">
    <div style="display:flex;align-items:baseline;gap:8px;flex-wrap:wrap;">
      <span style="font-size:16px;font-weight:700;color:#f1f5f9;
                   white-space:nowrap;overflow:hidden;text-overflow:ellipsis;">
        {title}
      </span>
      <span style="font-size:13px;color:#64748b;flex-shrink:0;">{year}</span>
    </div>
    <div style="margin:6px 0 8px;">{badges}</div>
    <div style="display:flex;align-items:center;gap:10px;flex-wrap:wrap;">
      <span style="color:#fbbf24;font-size:14px;letter-spacing:2px;">{stars}</span>
      <span style="color:#94a3b8;font-size:13px;">{avg_r:.1f}</span>
      <span style="color:#334155;font-size:12px;">|</span>
      <span style="color:#64748b;font-size:12px;">{num_fmt} ratings</span>
    </div>
  </div>

  <!-- Score pill -->
  <div style="background:#0f172a;border:1px solid #3b82f6;border-radius:10px;
              padding:8px 12px;text-align:center;flex-shrink:0;">
    <div style="font-size:18px;font-weight:800;color:#60a5fa;">{score:.2f}</div>
    <div style="font-size:9px;color:#64748b;text-transform:uppercase;
                letter-spacing:1px;margin-top:1px;">score</div>
  </div>

</div>""")

    cards.append('</div>')
    return '\n'.join(cards)

# ── Recommendation functions ──────────────────────────────────
def cold_start_recommend(preferred_genres, top_k, min_ratings_n, pop_weight):
    if not preferred_genres:
        return ('<div style="color:#ef4444;text-align:center;padding:32px;">'
                '⚠️ Please select at least one genre.</div>')

    user_vec = np.zeros(len(ALL_GENRES), dtype=np.float32)
    for g in preferred_genres:
        if g in ALL_GENRES:
            user_vec[ALL_GENRES.index(g)] = 1.0

    recs = []
    for mid, gvec in movie_id_to_genre.items():
        sim = float(np.dot(user_vec, gvec))
        if sim > 0 and mid in movie_stats.index:
            s = movie_stats.loc[mid]
            if s['num_ratings'] >= min_ratings_n:
                pop = float(np.log1p(s['num_ratings']) / np.log1p(max_num_ratings))
                score = sim * float(s['avg_rating']) * (1 + pop_weight * pop)
                recs.append({
                    'movieId':        mid,
                    'title':          movie_title_map.get(mid, 'Unknown'),
                    'genres':         movie_genre_map.get(mid, ''),
                    'avg_rating':     float(s['avg_rating']),
                    'num_ratings':    int(s['num_ratings']),
                    'combined_score': round(score, 3),
                })

    recs.sort(key=lambda x: x['combined_score'], reverse=True)
    for i, r in enumerate(recs[:int(top_k)], 1):
        r['rank'] = i

    return build_cards(recs[:int(top_k)], score_key='combined_score')


def gmf_recommend(user_id_str, top_k, min_ratings_n):
    try:
        uid = int(str(user_id_str).strip())
    except (ValueError, AttributeError):
        return ('<div style="color:#ef4444;text-align:center;padding:32px;">'
                '⚠️ Please enter a valid numeric User ID.</div>')

    if uid not in user_id_to_index:
        return (f'<div style="color:#ef4444;text-align:center;padding:32px;">'
                f'⚠️ User ID {uid} was not in the training set.  '
                f'Try the <b>New User</b> tab instead.</div>')

    u_idx = user_id_to_index[uid]
    candidates = [
        mid for mid in movie_id_to_index
        if mid in movie_stats.index
        and movie_stats.loc[mid, 'num_ratings'] >= min_ratings_n
    ]
    m_idxs = [movie_id_to_index[mid] for mid in candidates]

    all_scores = []
    with torch.no_grad():
        for i in range(0, len(m_idxs), 4096):
            bm = torch.tensor(m_idxs[i:i+4096], dtype=torch.long).to(device)
            bu = torch.tensor([u_idx] * len(bm), dtype=torch.long).to(device)
            preds = gmf_model(bu, bm).cpu().numpy()
            all_scores.extend((preds * (MAX_RATING - MIN_RATING) + MIN_RATING).tolist())

    recs = []
    for idx, mid in enumerate(candidates):
        recs.append({
            'movieId':    mid,
            'title':      movie_title_map.get(mid, 'Unknown'),
            'genres':     movie_genre_map.get(mid, ''),
            'avg_rating': float(movie_stats.loc[mid, 'avg_rating']),
            'num_ratings':int(movie_stats.loc[mid, 'num_ratings']),
            'gmf_score':  round(float(all_scores[idx]), 3),
        })

    recs.sort(key=lambda x: x['gmf_score'], reverse=True)
    for i, r in enumerate(recs[:int(top_k)], 1):
        r['rank'] = i

    return build_cards(recs[:int(top_k)], score_key='gmf_score')

# ── Gradio UI ─────────────────────────────────────────────────
CSS = """
body, .gradio-container { background:#0d1117 !important; }
.tab-nav { border-bottom: 1px solid #1e2d3d !important; }
.tab-nav button { font-weight:600; font-size:14px; }
footer { display:none !important; }
"""

HEADER = """
<div style="background:linear-gradient(135deg,#0f172a 0%,#1e2130 100%);
            border:1px solid #1e2d3d;border-radius:16px;
            padding:28px 24px 20px;margin-bottom:8px;text-align:center;">
  <h1 style="color:#f1f5f9;font-size:30px;font-weight:800;margin:0;letter-spacing:-0.5px;">
    🎬 Deep Learning Movie Recommender
  </h1>
  <p style="color:#94a3b8;font-size:14px;margin:8px 0 16px;">
    Generalized Matrix Factorization · MovieLens 25M · 17.4M training ratings
  </p>
  <div style="display:flex;justify-content:center;flex-wrap:wrap;gap:20px;">
    <div style="background:#0f172a;border:1px solid #1e3a5f;border-radius:10px;padding:8px 18px;">
      <div style="color:#60a5fa;font-size:20px;font-weight:800;">22,316</div>
      <div style="color:#64748b;font-size:11px;text-transform:uppercase;letter-spacing:1px;">movies</div>
    </div>
    <div style="background:#0f172a;border:1px solid #1e3a5f;border-radius:10px;padding:8px 18px;">
      <div style="color:#60a5fa;font-size:20px;font-weight:800;">121,673</div>
      <div style="color:#64748b;font-size:11px;text-transform:uppercase;letter-spacing:1px;">users</div>
    </div>
    <div style="background:#0f172a;border:1px solid #1e3a5f;border-radius:10px;padding:8px 18px;">
      <div style="color:#34d399;font-size:20px;font-weight:800;">0.8340</div>
      <div style="color:#64748b;font-size:11px;text-transform:uppercase;letter-spacing:1px;">RMSE</div>
    </div>
    <div style="background:#0f172a;border:1px solid #1e3a5f;border-radius:10px;padding:8px 18px;">
      <div style="color:#34d399;font-size:20px;font-weight:800;">93.3%</div>
      <div style="color:#64748b;font-size:11px;text-transform:uppercase;letter-spacing:1px;">hit rate</div>
    </div>
  </div>
</div>
"""

with gr.Blocks(css=CSS, theme=gr.themes.Base(
        primary_hue=gr.themes.colors.blue,
        secondary_hue=gr.themes.colors.purple,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "sans-serif"],
)) as demo:

    gr.HTML(HEADER)

    with gr.Tabs():

        # ── Tab 1: Cold Start ──────────────────────────────
        with gr.Tab("🆕  New User — Discover by Genre"):
            gr.Markdown(
                "Select the genres you enjoy and the model will surface "
                "the best-matching, highly-rated movies using the **V2 hybrid algorithm** "
                "(93.3% hit rate)."
            )
            with gr.Row(equal_height=False):
                with gr.Column(scale=1, min_width=280):
                    genre_cb = gr.CheckboxGroup(
                        choices=ALL_GENRES,
                        value=['Action', 'Adventure'],
                        label="Favourite genres",
                    )
                    top_k_cs   = gr.Slider(5, 25, value=10, step=1,
                                           label="Recommendations to show")
                    min_r_cs   = gr.Slider(10, 500, value=50, step=10,
                                           label="Minimum ratings (popularity filter)")
                    pop_w_cs   = gr.Slider(0.0, 0.5, value=0.2, step=0.05,
                                           label="Popularity boost  [ 0 = pure quality · 0.5 = more popular ]",
                                           info="0.2 gives the best hit rate in evaluation")
                    btn_cs = gr.Button("🎬  Get Recommendations", variant="primary", size="lg")

                with gr.Column(scale=2):
                    out_cs = gr.HTML(
                        value='<div style="color:#64748b;text-align:center;padding:60px;">'
                              'Your recommendations will appear here.</div>'
                    )

            btn_cs.click(
                fn=cold_start_recommend,
                inputs=[genre_cb, top_k_cs, min_r_cs, pop_w_cs],
                outputs=out_cs,
            )

        # ── Tab 2: Personalized GMF ────────────────────────
        with gr.Tab("👤  Returning User — Personalised Picks"):
            gr.Markdown(
                "Enter a User ID from the training set to get personalised recommendations "
                "scored directly by the **GMF neural model** (RMSE 0.8340 · 14.9% better than SVD)."
            )
            with gr.Row(equal_height=False):
                with gr.Column(scale=1, min_width=280):
                    uid_in     = gr.Textbox(label="User ID  (1 – 162,541)",
                                            placeholder="e.g.  42",
                                            value="1")
                    top_k_gmf  = gr.Slider(5, 25, value=10, step=1,
                                           label="Recommendations to show")
                    min_r_gmf  = gr.Slider(10, 500, value=50, step=10,
                                           label="Minimum ratings (popularity filter)")
                    btn_gmf = gr.Button("🎬  Get Personalised Recommendations",
                                        variant="primary", size="lg")

                with gr.Column(scale=2):
                    out_gmf = gr.HTML(
                        value='<div style="color:#64748b;text-align:center;padding:60px;">'
                              'Your recommendations will appear here.</div>'
                    )

            btn_gmf.click(
                fn=gmf_recommend,
                inputs=[uid_in, top_k_gmf, min_r_gmf],
                outputs=out_gmf,
            )

        # ── Tab 3: About ───────────────────────────────────
        with gr.Tab("ℹ️  About"):
            gr.Markdown("""
## About this Project

Built as a deep learning course project using the **MovieLens 25M** dataset —
25 million ratings from 162,541 users on 62,423 movies.

---

### Model Progression — RMSE on held-out test set

| Model | RMSE | Notes |
|-------|------|-------|
| Global Average | 1.0810 | Dumb baseline |
| Movie Average | 1.0186 | Per-movie mean |
| SVD | 0.9795 | Classical matrix factorisation |
| NCF (NeuBF-4) | 0.8430 | Neural CF, 4 layers |
| GMF Original | 0.8394 | Generalized matrix factorisation |
| MLP | 0.8368 | Multilayer perceptron |
| **GMF Regularised** | **0.8340** | ← best model, used here |
| NeuMF | 0.8450 | Unstable on this dataset |

**14.85% improvement over SVD.**

---

### New-User Algorithm (Cold Start V2)

Activated for users without rating history.

```
popularity  = log1p(num_ratings) / log1p(max_ratings)   # 0–1
score       = genre_match × avg_rating × (1 + 0.2 × popularity)
```

Evaluation on 1,000 held-out cold-start users:

| System | Hit Rate@10 |
|--------|-------------|
| Popularity baseline | 92.25% |
| V1 — quality only | 88.93% |
| **V2 — quality + popularity** | **93.31%** |
| V3 — weighted genres | 90.88% |

---

### GMF Architecture

```
user_id  →  Embedding(121673, 32)  ┐
                                    ⊗  →  BN  →  Dropout(0.3)  →  Linear(32,1)
movie_id →  Embedding(22316,  32)  ┘
```

- Embeddings initialised with N(0, 0.1)
- Adam optimiser, weight-decay regularisation
- Trained on 17.4M ratings (temporal split — pre-2015)

---

### A/B Test Summary (2,000 users per group)

No statistically significant difference between V2 and baseline — all p-values > 0.05,
confirming that offline metric gains do not always translate to live improvements.

---

*Dataset: [MovieLens 25M](https://grouplens.org/datasets/movielens/25m/)*
            """)

demo.launch(share=True)
